# nano-prover: train on Kaggle (free T4), verify locally

**Before running — right-hand panel:**
1. **Settings → Accelerator → `GPU T4 x2`**
2. **Settings → Internet → On**  (needed to clone + pip install + download the model)

Edit the **CONFIG** at the top of cell 1 and the **TOKEN** in cell 2, then run both cells.

This config (1.5B · 3 epochs · 1024-token gen · pass@3) takes **~3 h** on a T4 — well within Kaggle's
9 h/session and 30 h/week. For a quick ~1 h sanity run set `MODEL='…0.5B…'`, `EPOCHS=1`, `K=1`.
Changes vs the 1st run: **1024-token generations** (last time 59% were truncated at 512), the
**1.5B** base, and **3 epochs** — the three levers the 1st report recommended.

In [ ]:
# ================= CONFIG =================
MODEL  = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'   # quick run: 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
EPOCHS = 3
MAXTOK = 1024    # generation length cap (512 truncated 59% of proofs last time)
K      = 3       # samples per problem -> pass@K
# ==========================================
!nvidia-smi -L
import os
os.chdir('/kaggle/working')
!rm -rf Conjecture-Gym
!git clone --depth 1 https://github.com/FELIXGAO0315/Conjecture-Gym.git
os.chdir('/kaggle/working/Conjecture-Gym')
!pip install -q -U transformers trl peft datasets accelerate
!python training/scripts/prepare_data.py
!SFT_MODEL={MODEL} SFT_EPOCHS={EPOCHS} SFT_GRAD_CKPT=1 python training/scripts/sft_train.py
!python training/scripts/gen_preds.py --n 100 --k {K} --max-new-tokens {MAXTOK}

In [ ]:
# push the small preds.jsonl back to GitHub.  EDIT THE TOKEN (keep the quotes).
#   github.com -> Settings -> Developer settings -> Personal access tokens -> Tokens (classic)
#   -> Generate new token -> scope: repo -> copy the ghp_... string.
TOKEN = 'ghp_PUT_YOUR_REAL_TOKEN_HERE'
import os
os.chdir('/kaggle/working/Conjecture-Gym')
!git add -f training/scripts/preds.jsonl
!git -c user.email=you@example.com -c user.name=you commit -m 'eval preds'
!git push https://{TOKEN}@github.com/FELIXGAO0315/Conjecture-Gym.git main

Then on your **local** machine (where the Lean verifier lives):
```bash
git pull
python training/assessment/assessment.py     # Lean-verified pass@k + full report → training/output/
```